# Sub-etapa 2.1 (preparación) · Frecuencia dominante en trozos de 8 s frente a la caminata completa

En su correo del 9 de noviembre de 2026 (punto 2), el profesor guía pidió que, antes de fijar los umbrales de concordancia de la Etapa 2, se simulara con las señales limpias de LTMM **cuánto varía la frecuencia dominante de la misma IMU en trozos de 8 s frente a la caminata completa**. En su prueba rápida, la diferencia superó 0,10 Hz en cerca del 8 % de los trozos y en un 2 % el pico saltó a la mitad o al doble de la frecuencia de paso.

Este notebook hace esa simulación y prueba la **regla de errores gruesos** del plan de la Etapa 2 (sección 4, punto 3). No usa datos nuevos ni modifica los notebooks anteriores.

**Sobre los números del plan.** La tabla de la sección 4 del plan (versión 3) se calculó con una primera versión de este código que no quedó guardada en el repositorio. Este notebook la reemplaza: si algún valor difiere, vale el de este notebook y se corrige la tabla del plan.

**Antes de ejecutarlo:** deben existir las señales limpias en `datos/procesados/senales_limpias/` (las genera el notebook 01).

## Plan de análisis (escrito antes de ejecutar)

Este notebook se sube a GitHub **sin ejecutar** antes de correrlo, igual que el 05b y el 05c.

1. **Trozos.** Cada señal limpia (`acc_v`, 100 Hz) se corta en trozos de 8 s consecutivos, sin superposición. El resto que no completa 8 s se descarta. 8 s es lo que dura, aproximadamente, una pasada de 10 m.
2. **Frecuencia dominante.** Se calcula con la función `espectro()` del notebook 01, copiada sin cambios, en cada trozo y en la caminata completa.
3. **Regla de errores gruesos.** La misma del plan y de la aplicación de registro (`frecuencia_con_control()`): se cuentan los pasos, y si la frecuencia dominante queda fuera de 0,75 a 1,33 veces la cadencia contada, se reemplaza por el pico del espectro más cercano a esa cadencia. Con menos de 3 pasos el trozo se descarta.
4. **Qué se informa**, sin la regla y con la regla (en el segundo caso la regla se aplica al trozo y también a la caminata completa):
   * porcentaje de trozos con diferencia mayor que 0,10 Hz y mayor que 0,05 Hz;
   * porcentaje de trozos en que el pico saltó a la mitad o al doble (razón trozo / caminata entre 0,4 y 0,6, o entre 1,8 y 2,2);
   * límites de concordancia del 95 % (media ± 1,96 desviaciones estándar de la diferencia);
   * cuántos trozos corrigió la regla y en qué caminatas completas actuó.
5. **Qué no se hace.** No se cambian los umbrales del plan según este resultado: sirve para saber cuánta variación aporta la ventana corta por sí sola, y para decidir sobre qué medición se aplica el umbral de 1,93 Hz.

## 1. Configuración

In [ ]:
import platform
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import signal

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SENALES = RAIZ / "datos" / "procesados" / "senales_limpias"
RESULTADOS = RAIZ / "resultados"
FIGURAS = RAIZ / "figuras"

FS = 100                # Hz, frecuencia de muestreo de LTMM
TROZO_S = 8             # duración de cada trozo
RAZON_MIN, RAZON_MAX = 0.75, 1.33   # regla de errores gruesos (plan, sección 4)
MIN_PASOS = 3

print(datetime.now().strftime("%Y-%m-%d %H:%M"), "| Python", platform.python_version(),
      "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)
archivos = sorted(SENALES.glob("*_base.csv"))
print(len(archivos), "señales limpias")

## 2. Funciones

`espectro()` viene del notebook 01, sin cambios. `contar_pasos()` y `frecuencia_con_control()` son las de la aplicación de registro de la Etapa 2 (`imu_marcha_pc/analisis.py` en el repositorio imu-marcha-app), para que la regla que se prueba aquí sea exactamente la que se usará al medir.

In [ ]:
def espectro(v, fs):
    f, p = signal.welch(v, fs=fs, nperseg=min(len(v), int(10 * fs)), nfft=8192)
    banda = (f >= 0.5) & (f <= 3.0)
    i = np.argmax(np.where(banda, p, 0))
    mitad = p[i] / 2
    izq = i
    while izq > 0 and p[izq] > mitad:
        izq -= 1
    der = i
    while der < len(p) - 1 and p[der] > mitad:
        der += 1
    return f[i], p[i], f[der] - f[izq]


def pasabajo(x, fs, fc, orden=4):
    b, a = signal.butter(orden, fc / (fs / 2), btype="low")
    return signal.filtfilt(b, a, x, axis=0)


def contar_pasos(v, fs):
    # Cadencia contando pasos: picos de la señal suavizada a 3 Hz
    suave = pasabajo(v, fs, 3)
    picos, _ = signal.find_peaks(suave, distance=int(0.3 * fs), prominence=0.3 * np.std(suave))
    if len(picos) < 3:
        return np.nan, len(picos)
    return 60 / np.median(np.diff(picos) / fs), len(picos)


def frecuencia_con_control(v, fs):
    # Frecuencia dominante + regla de errores gruesos
    fd, _, _ = espectro(v, fs)
    cad, n = contar_pasos(v, fs)
    res = {"frec_original_hz": fd, "frec_hz": fd, "cadencia_pasos_min": cad, "n_pasos": n,
           "corregida": False, "descartada": False}
    if n < MIN_PASOS or not np.isfinite(cad):
        res["descartada"] = True
        return res
    razon = fd * 60 / cad
    if not (RAZON_MIN <= razon <= RAZON_MAX):
        f, p = signal.welch(v, fs=fs, nperseg=min(len(v), int(10 * fs)), nfft=8192)
        banda = (f >= 0.5) & (f <= 3.0)
        picos, _ = signal.find_peaks(np.where(banda, p, 0))
        if len(picos):
            k = picos[np.argmin(np.abs(f[picos] - cad / 60))]
            res["frec_hz"] = float(f[k])
            res["corregida"] = True
    return res

## 3. Caminata completa y trozos de 8 s

In [ ]:
filas, completas = [], []
n_trozo = TROZO_S * FS
for arch in archivos:
    persona = arch.stem.replace("_base", "")
    v = pd.read_csv(arch)["acc_v"].to_numpy()
    c = frecuencia_con_control(v, FS)
    completas.append({"persona": persona, "duracion_s": len(v) / FS, **c})
    for k, i in enumerate(range(0, len(v) - n_trozo + 1, n_trozo)):
        t = frecuencia_con_control(v[i:i + n_trozo], FS)
        filas.append({"persona": persona, "trozo": k,
                      "completa_sin_regla_hz": c["frec_original_hz"], "completa_con_regla_hz": c["frec_hz"],
                      "trozo_sin_regla_hz": t["frec_original_hz"], "trozo_con_regla_hz": t["frec_hz"],
                      "cadencia_trozo_pasos_min": t["cadencia_pasos_min"], "n_pasos": t["n_pasos"],
                      "corregido": t["corregida"], "descartado": t["descartada"]})

completas = pd.DataFrame(completas)
trozos = pd.DataFrame(filas)
print(len(completas), "caminatas,", len(trozos), "trozos de", TROZO_S, "s,",
      int(trozos["descartado"].sum()), "descartados por tener menos de", MIN_PASOS, "pasos")
trozos.head()

## 4. Resultado: sin la regla y con la regla

In [ ]:
def es_salto(razon):
    # el pico quedó en la mitad o en el doble de la frecuencia de la caminata completa
    return ((razon > 0.4) & (razon < 0.6)) | ((razon > 1.8) & (razon < 2.2))


def resumen(trozo, completa):
    d = trozo - completa
    lim = d.mean() - 1.96 * d.std(ddof=1), d.mean() + 1.96 * d.std(ddof=1)
    return {"trozos": len(d),
            "dif_mayor_0,10_Hz_%": 100 * (d.abs() > 0.10).mean(),
            "dif_mayor_0,05_Hz_%": 100 * (d.abs() > 0.05).mean(),
            "salto_mitad_o_doble_%": 100 * es_salto(trozo / completa).mean(),
            "sesgo_hz": d.mean(), "limite_inf_hz": lim[0], "limite_sup_hz": lim[1]}


validos = trozos[~trozos["descartado"]]
tabla = pd.DataFrame({
    "Sin la regla": resumen(validos["trozo_sin_regla_hz"], validos["completa_sin_regla_hz"]),
    "Con la regla": resumen(validos["trozo_con_regla_hz"], validos["completa_con_regla_hz"]),
}).T
tabla["trozos_corregidos_%"] = [np.nan, 100 * validos["corregido"].mean()]
tabla.round(2)

### Caminatas completas en que actuó la regla

Si la regla corrige una caminata completa, quiere decir que el pico del espectro de esa persona quedó en la frecuencia de zancada (la mitad) o en un armónico, y no en la frecuencia de paso.

In [ ]:
completas.loc[completas["corregida"], ["persona", "duracion_s", "frec_original_hz", "frec_hz",
                                       "cadencia_pasos_min", "n_pasos"]].round(2)

## 5. Figura

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for eje, col_t, col_c, titulo in [
        (ejes[0], "trozo_sin_regla_hz", "completa_sin_regla_hz", "Sin la regla"),
        (ejes[1], "trozo_con_regla_hz", "completa_con_regla_hz", "Con la regla de errores gruesos")]:
    eje.scatter(validos[col_c], validos[col_t], s=12, alpha=0.5)
    lim = [0.5, 3.0]
    eje.plot(lim, lim, color="gray", lw=1)
    eje.plot(lim, [x / 2 for x in lim], color="gray", lw=0.8, ls="--")
    eje.plot(lim, [x * 2 for x in lim], color="gray", lw=0.8, ls="--")
    eje.set_xlim(1.0, 3.0); eje.set_ylim(0.5, 3.0)
    eje.set_xlabel("Caminata completa (Hz)"); eje.set_title(titulo)
ejes[0].set_ylabel("Trozo de 8 s (Hz)")
fig.suptitle("Frecuencia dominante: trozo de 8 s frente a la caminata completa (líneas punteadas: mitad y doble)")
fig.tight_layout()
FIGURAS.mkdir(exist_ok=True)
fig.savefig(FIGURAS / "06_trozos_8s_vs_completa.png", dpi=150)
plt.show()

## 6. Guardar resultados

In [ ]:
RESULTADOS.mkdir(exist_ok=True)
trozos.round(4).to_csv(RESULTADOS / "simulacion_trozos_8s.csv", index=False)
tabla.round(3).to_csv(RESULTADOS / "simulacion_trozos_8s_resumen.csv")
completas.round(4).to_csv(RESULTADOS / "simulacion_caminata_completa_regla.csv", index=False)
print("Guardado en", RESULTADOS)

## 7. Lectura de los resultados

*(Completar después de ejecutar, con los números de la tabla de la sección 4.)*

Preguntas que debe responder esta sección:

1. ¿Qué porcentaje de trozos de 8 s difiere en más de 0,10 Hz de la caminata completa? ¿Coincide con la prueba rápida del profesor (cerca del 8 %)?
2. ¿Cuánto reduce la regla los saltos a la mitad o al doble?
3. ¿Los límites de concordancia entre el trozo y la caminata completa caben en ±0,10 Hz? Si no caben, la variación viene de la ventana corta y no de la cámara: por eso la comparación principal de la Etapa 2 es cámara contra IMU **en la misma ventana**, y el umbral de 1,93 Hz se aplica sobre la caminata de 1 minuto.
4. ¿En qué caminatas completas actuó la regla, y qué implica para el umbral de la Etapa 1?